# 02 - Preprocessing

Apply cleaning steps from `src/data/clean_data.py`: drop sparse/ID columns, drop encounters where the patient died (can't be "readmitted"), keep one encounter per patient (leakage guard), bucket the ICD-9 diagnosis codes into categories, and build the binary 30-day target. Save the result for modeling.

In [1]:
%load_ext autoreload
%autoreload 2

from src.config import PROCESSED_DATA_DIR, TARGET_COLUMN
from src.data.load_data import load_raw_data
from src.data.clean_data import clean

df = load_raw_data()
df.shape

(101766, 50)

In [2]:
df_clean = clean(df)
df_clean.shape

(70439, 45)

### Why the row count drops

- Encounters where the patient died during the stay are dropped (they cannot be readmitted — keeping them would just add label noise).
- Only the **first encounter per patient** is kept. Multiple rows can belong to the same `patient_nbr`; if a later encounter for the same patient ended up in the test set while an earlier one was in training, the model could partly "recognize" the patient rather than generalize — a classic train/test leakage mistake worth knowing for interviews.

In [3]:
from src.data.load_data import load_raw_data

raw = load_raw_data()
print(f"raw rows: {len(raw):,} | unique patients: {raw['patient_nbr'].nunique():,}")
print(f"clean rows: {len(df_clean):,}")

raw rows: 101,766 | unique patients: 71,518
clean rows: 70,439


### Diagnosis categories

`diag_1/2/3` are raw ICD-9 codes (hundreds of distinct values) — too high-cardinality to one-hot encode usefully. `clean()` buckets them into clinical categories instead.

In [4]:
df_clean["diag_1_cat"].value_counts()

diag_1_cat
circulatory        21499
other              12184
respiratory         9566
digestive           6513
diabetes            5766
injury              4705
musculoskeletal     4071
genitourinary       3477
neoplasms           2658
Name: count, dtype: int64

In [5]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_DATA_DIR / "diabetic_data_clean.csv", index=False)
print(f"saved {len(df_clean):,} rows, {df_clean.shape[1]} columns")

saved 70,439 rows, 45 columns
